# Qudor AlphaZero training
This notebook runs the current MCTS-based pipeline. Outputs stay in Google Drive and resume automatically after a Colab disconnect.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), 'VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    print('No accelerator. Use the CPU cell below, or pick Runtime -> Change runtime type -> T4 GPU.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Qudor
!python -m pip install -q -e .
!PYTHONPATH=. python scripts/benchmark.py

## GPU validation
Run this once after copying the project to Drive. It confirms MCTS self-play, training, checkpoint writing and arena gating.

In [ ]:
!python -m quoridor_ai.az_train --config configs/colab_az_t4_fast.json --output /content/drive/MyDrive/Qudor_runs/az_t4_fast

## Main 15 GB run
Rerun this exact cell after any disconnect. It reads `latest.pt` and continues; do not add `--no-resume`.

In [ ]:
# Gumbel AlphaZero: ~24 sims/move instead of 192, same net. Rerun as-is after any
# disconnect - it resumes from latest.pt in the same output directory.
!python -m quoridor_ai.az_train --config configs/colab_az_gumbel.json --output /content/drive/MyDrive/Qudor_runs/az_15gb


## No GPU available

Colab's limits are on the accelerator, not the runtime. Set **Runtime -> Change runtime type -> None** and run the cell below: it continues the *same* run from the same directory on CPU, at roughly 2% of the GPU's rate, without spending any GPU quota.

Switch back to a GPU later and just rerun the main cell above.

In [ ]:
# Same network, same directory, no accelerator. Resumes latest.pt like any other restart.
!python -m quoridor_ai.az_train --config configs/colab_az_cpu.json --output /content/drive/MyDrive/Qudor_runs/az_15gb

In [ ]:
import pandas as pd
from pathlib import Path
p = Path('/content/drive/MyDrive/Qudor_runs/az_15gb/metrics.csv')
if p.exists():
    df = pd.read_csv(p)
    display(df.tail(20))
    df.plot(x='iteration', y=['total_loss', 'games_per_sec', 'positions_per_sec', 'gate_win_rate'], subplots=True, figsize=(13, 11), grid=True)
else:
    print('Run the main training cell first.')